# Cài đặt thư viện cần thiết

In [1]:
!pip uninstall -y transformers tokenizers
!pip install -q transformers==4.41.2 tokenizers==0.19.1 sentencepiece protobuf

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 89.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.0 MB/s eta 0:00:00


In [2]:
!pip install -q pyvi emoji transformers scikit-learn openpyxl accelerate

import os
import json
import re
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, AutoConfig, get_scheduler
from sklearn.metrics import f1_score, classification_report
from pyvi.ViTokenizer import tokenize
import emoji
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel, AutoConfig, get_scheduler, T5EncoderModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

os.makedirs("/kaggle/working/saved_models", exist_ok=True)
os.makedirs("/kaggle/working/reports", exist_ok=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 59.3 MB/s eta 0:00:00
Device: cuda


# Khai báo thư viện và kiểm tra GPU

In [3]:
import subprocess

REPO_URL = "https://github.com/ricardo-tran/ViGoEmotions.git"
REPO_DIR = "/kaggle/working/ViGoEmotions_Original"

if not os.path.exists(REPO_DIR):
    print("Cloning repo...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    print("Repo already exists")

DOCS_PATH = os.path.join(REPO_DIR, "model", "docs")
CORPUS_PATH = os.path.join(REPO_DIR, "corpus")

# Load dictionaries
with open(os.path.join(DOCS_PATH, "patterns.json"), encoding="utf-8") as f:
    pattern_dict = json.load(f)

with open(os.path.join(DOCS_PATH, "emojis.json"), encoding="utf-8") as f:
    emoji_dict = json.load(f)

teen_dict = {}
with open(os.path.join(DOCS_PATH, "teencode4.txt"), encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line and "\t" in line:
            old, new = line.split("\t", 1)
            teen_dict[old] = new

print("Dictionaries loaded")

# Load dataset
excel_path = os.path.join(CORPUS_PATH, "dataset_V1.xlsx")
excel_file = pd.ExcelFile(excel_path)

if "train" in excel_file.sheet_names:
    train_df = pd.read_excel(excel_file, sheet_name="train")
    val_df   = pd.read_excel(excel_file, sheet_name="val")
    test_df  = pd.read_excel(excel_file, sheet_name="test")
else:
    df = pd.read_excel(excel_file, sheet_name="Sheet1")
    train_df = df[df["set"] == "train"].copy()
    val_df   = df[df["set"] == "val"].copy()
    test_df  = df[df["set"] == "test"].copy()

print(f"Train: {train_df.shape} | Val: {val_df.shape} | Test: {test_df.shape}")

Cloning repo...


Cloning into '/kaggle/working/ViGoEmotions_Original'...


Dictionaries loaded
Train: (16531, 3) | Val: (2066, 3) | Test: (2067, 3)


# Cấu hình đường dẫn và Tải Dữ liệu (Dictionaries & Dataset) 

In [4]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()

    # 1. Normalize patterns
    for pattern, replacement in pattern_dict.items():
        text = re.sub(pattern, replacement, text)

    # 2. Remove duplicate alphabet characters
    result, prev = [], None
    for char in text:
        if char.isalpha() and char == prev:
            continue
        prev = char
        result.append(char)
    text = "".join(result)

    # 3. Remove duplicate emojis
    result, prev_emoji = [], None
    for char in text:
        if char in emoji.EMOJI_DATA:
            if char == prev_emoji:
                continue
            prev_emoji = char
        else:
            prev_emoji = None
        result.append(char)
    text = "".join(result)

    # 4. Replace teencode
    for old, new in teen_dict.items():
        text = re.sub(rf"\b{re.escape(old)}\b", new, text)

    # 5. Replace emojis
    for emj, rep in emoji_dict.items():
        text = text.replace(emj, f" {rep} ")

    # 6. Format punctuation & whitespace
    text = re.sub(r"(?<![.,!?;:])\n", ". ", text)
    text = re.sub(r"\n([.,!?;:])?", r" \1", text)
    text = re.sub(r"([.,!?;:])", r" \1 ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

print("Applying S2 preprocessing...")
for df in [train_df, val_df, test_df]:
    df["text"] = df["text"].astype(str).apply(clean_text)

print("Preprocessing done")
print(train_df["text"].iloc[0])

Applying S2 preprocessing...
Preprocessing done
xem mà ngẫm lại cuộc đời bản thân ta đã trải qua nhiều thứ ta rồi cũng sẽ lớn kí ước sẽ còn mãi trong lòng


In [5]:
!find /kaggle/working/ViGoEmotions_Original -maxdepth 3 -type f

/kaggle/working/ViGoEmotions_Original/.git/index
/kaggle/working/ViGoEmotions_Original/.git/info/exclude
/kaggle/working/ViGoEmotions_Original/.git/packed-refs
/kaggle/working/ViGoEmotions_Original/.git/HEAD
/kaggle/working/ViGoEmotions_Original/.git/logs/HEAD
/kaggle/working/ViGoEmotions_Original/.git/description
/kaggle/working/ViGoEmotions_Original/.git/hooks/pre-receive.sample
/kaggle/working/ViGoEmotions_Original/.git/hooks/pre-applypatch.sample
/kaggle/working/ViGoEmotions_Original/.git/hooks/push-to-checkout.sample
/kaggle/working/ViGoEmotions_Original/.git/hooks/applypatch-msg.sample
/kaggle/working/ViGoEmotions_Original/.git/hooks/post-update.sample
/kaggle/working/ViGoEmotions_Original/.git/hooks/fsmonitor-watchman.sample
/kaggle/working/ViGoEmotions_Original/.git/hooks/pre-push.sample
/kaggle/working/ViGoEmotions_Original/.git/hooks/prepare-commit-msg.sample
/kaggle/working/ViGoEmotions_Original/.git/hooks/commit-msg.sample
/kaggle/working/ViGoEmotions_Original/.git/hooks/pr

# Tiền xử lý văn bản (S2 Preprocessing)

In [6]:
with open(os.path.join(DOCS_PATH, "label_dict.json"), encoding="utf-8") as f:
    label_dict = json.load(f)

label_to_idx = {label: int(idx) for idx, label in label_dict.items()}
print("Number of labels:", len(label_dict))

def encode_labels(label_str, label_dict):
    labels = str(label_str).replace("[", "").replace("]", "").replace("'", "").replace('"', "").split(",")
    labels = [x.strip() for x in labels if x.strip()]
    vec = np.zeros(len(label_dict), dtype=np.float32)

    if labels and labels[0].isnumeric():
        labels = [int(x) for x in labels]
        for idx in label_dict.values():
            if idx in labels:
                vec[idx] = 1.0
    else:
        for lab, idx in label_dict.items():
            if lab in labels:
                vec[idx] = 1.0
    return vec

train_texts  = train_df["text"].tolist()
train_labels = [encode_labels(x, label_to_idx) for x in train_df["labels"]]
val_texts    = val_df["text"].tolist()
val_labels   = [encode_labels(x, label_to_idx) for x in val_df["labels"]]
test_texts   = test_df["text"].tolist()
test_labels  = [encode_labels(x, label_to_idx) for x in test_df["labels"]]

print("Labels encoded")
print("Example label:", train_labels[0])

Number of labels: 28
Labels encoded
Example label: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0.]


# Load Labels và Mã hóa One-hot (Label Encoding)

In [7]:
!pip install -q sentencepiece protobuf huggingface_hub

from huggingface_hub import hf_hub_download
from transformers import T5Tokenizer
from pyvi.ViTokenizer import tokenize

model_type = "vit5"
model_name = "VietAI/vit5-base"
max_len = 200
BATCH_SIZE = 8

spiece_path = hf_hub_download(repo_id=model_name, filename="spiece.model")
tokenizer = T5Tokenizer(
    vocab_file=spiece_path,
    extra_ids=100,
    legacy=True,
    model_max_length=512,
)

tokenizer.name_or_path = model_name

print("Tokenizer:", type(tokenizer).__name__)
print("Vocab size:", tokenizer.vocab_size) 
assert tokenizer.vocab_size > 10000, f"Vocab sai: {tokenizer.vocab_size} — đừng train tiếp"

class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=200):
        self.texts = texts
        self.labels = torch.tensor(np.array(labels), dtype=torch.float32)
        self.tokenizer = tokenizer
        self.max_len = max_len

        self.prefix = ""
        name = getattr(tokenizer, "name_or_path", "").lower()
        if "vit5" in name:
            self.prefix = "classification: "

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        text = tokenize(text)      
        text = self.prefix + text  

        encoding = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
            return_tensors="pt",
            return_attention_mask=True,
        )
        return {
            "text": text,
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "targets": self.labels[idx],
        }

train_dataset = SentimentDataset(train_texts, train_labels, tokenizer, max_len)
val_dataset   = SentimentDataset(val_texts, val_labels, tokenizer, max_len)
test_dataset  = SentimentDataset(test_texts, test_labels, tokenizer, max_len)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("DataLoader ready (ViT5)")
print("Train batches:", len(train_loader))

spiece.model:   0%|          | 0.00/820k [00:00<?, ?B/s]

Tokenizer: T5Tokenizer
Vocab size: 36000
DataLoader ready (ViT5)
Train batches: 2067


# Khởi tạo Tokenizer và DataLoader

In [8]:
class ModelSentimentClassifier(nn.Module):
    def __init__(self, n_classes, model_type="vit5"):
        super().__init__()
        self.model_type = model_type

        if model_type == "vit5":
            model_name = "VietAI/vit5-base"
        elif model_type == "vit5-l":
            model_name = "VietAI/vit5-large"
        else:
            raise ValueError(model_type)

        self.backbone = T5EncoderModel.from_pretrained(model_name)
        self.drop = nn.Dropout(0.2)

        hidden = getattr(self.backbone.config, "hidden_size", None) or self.backbone.config.d_model
        self.fc = nn.Linear(hidden, n_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )
        pooled = outputs.last_hidden_state[:, 0, :]
        x = self.drop(pooled)
        return {"logits": self.fc(x)}

model = ModelSentimentClassifier(n_classes=len(label_dict), model_type=model_type).to(device)
print("ViT5 (T5EncoderModel) loaded")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/904M [00:00<?, ?B/s]

ViT5 (T5EncoderModel) loaded
Parameters: 112,697,500


# Cấu hình

In [9]:
EPOCHS = 12
optimizer = AdamW(model.parameters(), lr=5e-5)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=len(train_loader),
    num_training_steps=len(train_loader) * EPOCHS
)

label_counts = np.sum(train_labels, axis=0)
pos_weight = torch.tensor(
    [(len(train_labels) - c) / max(c, 1) for c in label_counts],
    dtype=torch.float32
).to(device)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

def run_epoch(model, loader, is_train=True):
    model.train() if is_train else model.eval()
    losses, all_y, all_p = [], [], []

    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        for batch in tqdm(loader, leave=False):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            targets = batch["targets"].to(device)

            if is_train:
                optimizer.zero_grad()

            logits = model(input_ids, attention_mask)["logits"]
            loss = loss_fn(logits, targets)
            losses.append(loss.item())

            if is_train:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                lr_scheduler.step()

            preds = (torch.sigmoid(logits) >= 0.5).int()
            all_y.append(targets.cpu().numpy())
            all_p.append(preds.cpu().numpy())

    y = np.vstack(all_y)
    p = np.vstack(all_p)
    f1 = f1_score(y, p, average="macro", zero_division=0)
    return np.mean(losses), f1

best_f1 = 0.0
history = {
    "epoch": [],
    "train_loss": [], "train_f1": [],
    "val_loss": [], "val_f1": []
}

print("Start training ViT5...")
for epoch in range(1, EPOCHS + 1):
    print(f"\n===== Epoch {epoch}/{EPOCHS} =====")
    train_loss, train_f1 = run_epoch(model, train_loader, is_train=True)
    val_loss, val_f1 = run_epoch(model, val_loader, is_train=False)

    history["epoch"].append(epoch)
    history["train_loss"].append(train_loss)
    history["train_f1"].append(train_f1)
    history["val_loss"].append(val_loss)
    history["val_f1"].append(val_f1)

    print(f"Train Loss: {train_loss:.4f} | Train Macro-F1: {train_f1:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Macro-F1: {val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), f"/kaggle/working/saved_models/{model_type}_best.pth")
        print(f"Saved best model (Val F1 = {best_f1:.4f})")

# Lưu history
history_df = pd.DataFrame(history)
history_df.to_excel(f"/kaggle/working/reports/metrics_{model_type}.xlsx", index=False)
print("\nTraining finished!")
print(f"Metrics saved to: /kaggle/working/reports/metrics_{model_type}.xlsx")

Start training ViT5...

===== Epoch 1/12 =====


Train Loss: 1.1180 | Train Macro-F1: 0.1934
Val   Loss: 0.8093 | Val   Macro-F1: 0.4031
Saved best model (Val F1 = 0.4031)

===== Epoch 2/12 =====


Train Loss: 0.7060 | Train Macro-F1: 0.4085
Val   Loss: 0.6767 | Val   Macro-F1: 0.4256
Saved best model (Val F1 = 0.4256)

===== Epoch 3/12 =====


Train Loss: 0.5218 | Train Macro-F1: 0.5078
Val   Loss: 0.7259 | Val   Macro-F1: 0.5120
Saved best model (Val F1 = 0.5120)

===== Epoch 4/12 =====


Train Loss: 0.3896 | Train Macro-F1: 0.5952
Val   Loss: 0.7748 | Val   Macro-F1: 0.5132
Saved best model (Val F1 = 0.5132)

===== Epoch 5/12 =====


...

Train Loss: 0.1532 | Train Macro-F1: 0.8227
Val   Loss: 1.2666 | Val   Macro-F1: 0.5841
Saved best model (Val F1 = 0.5841)

===== Epoch 9/12 =====


Train Loss: 0.1208 | Train Macro-F1: 0.8616
Val   Loss: 1.3749 | Val   Macro-F1: 0.5863
Saved best model (Val F1 = 0.5863)

===== Epoch 10/12 =====


Train Loss: 0.0960 | Train Macro-F1: 0.8913
Val   Loss: 1.4991 | Val   Macro-F1: 0.5972
Saved best model (Val F1 = 0.5972)

===== Epoch 11/12 =====


Train Loss: 0.0768 | Train Macro-F1: 0.9142
Val   Loss: 1.5519 | Val   Macro-F1: 0.5947

===== Epoch 12/12 =====


Train Loss: 0.0633 | Train Macro-F1: 0.9301
Val   Loss: 1.6137 | Val   Macro-F1: 0.6006
Saved best model (Val F1 = 0.6006)

Training finished!
Metrics saved to: /kaggle/working/reports/metrics_vit5.xlsx


# Thiết lập Hàm Huấn luyện & Đánh giá

In [10]:
# Load best model
model.load_state_dict(torch.load(f"/kaggle/working/saved_models/{model_type}_best.pth"))
model.eval()

all_targets, all_preds = [], []

with torch.no_grad():
    for batch in tqdm(test_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        targets = batch["targets"].to(device)

        logits = model(input_ids, attention_mask)["logits"]
        preds = (torch.sigmoid(logits) >= 0.5).int()

        all_targets.append(targets.cpu().numpy())
        all_preds.append(preds.cpu().numpy())

y_true = np.vstack(all_targets)
y_pred = np.vstack(all_preds)

macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
micro_f1 = f1_score(y_true, y_pred, average="micro", zero_division=0)

print("\n TEST RESULTS (ViT5) ")
print(f"Macro F1: {macro_f1:.4f}")
print(f"Micro F1: {micro_f1:.4f}")

# Classification report
report = classification_report(
    y_true, y_pred,
    target_names=list(label_dict.values()),
    zero_division=0,
    output_dict=True
)
report_df = pd.DataFrame(report).transpose()
report_df.to_excel(f"/kaggle/working/reports/classification_report_{model_type}.xlsx", index=True)

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=list(label_dict.values()), zero_division=0))
print(f"\nReport saved to: /kaggle/working/reports/classification_report_{model_type}.xlsx")

100%|██████████| 259/259 [00:27<00:00,  9.49it/s]



 TEST RESULTS (ViT5) 
Macro F1: 0.6022
Micro F1: 0.6171

Classification Report:
                precision    recall  f1-score   support

     amusement       0.68      0.81      0.74       374
    excitement       0.52      0.56      0.54        98
           joy       0.52      0.69      0.59       204
          love       0.61      0.76      0.68       143
        desire       0.38      0.56      0.45        80
      optimism       0.65      0.78      0.71       142
        caring       0.58      0.73      0.65       150
         pride       0.64      0.67      0.66        86
    admiration       0.52      0.66      0.58       101
     gratitude       0.86      0.89      0.87       108
        relief       0.51      0.70      0.59        60
      approval       0.59      0.65      0.62       115
   realization       0.44      0.49      0.46        95
      surprise       0.57      0.61      0.59        85
     curiosity       0.61      0.67      0.64       100
     confusion       0

# train loop